[![Github](https://img.shields.io/github/stars/labmlai/annotated_deep_learning_paper_implementations?style=social)](https://github.com/labmlai/annotated_deep_learning_paper_implementations)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labmlai/annotated_deep_learning_paper_implementations/blob/master/labml_nn/transformers/gpt/experiment.ipynb)                    

## Training a model with GPT architecture

This is an experiment training Tiny Shakespeare dataset with GPT architecture model.

Install the `labml-nn` package

In [4]:
!pip install labml-nn

  Using cached labml_nn-0.5.1-py3-none-any.whl.metadata (9.1 kB)
  Using cached labml-0.5.3-py3-none-any.whl.metadata (7.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.9/461.9 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.2 MB/s eta 0:00:00
  Created wheel for fairscale: filename=fairscale-0.4.13-py3-none-any.whl size=332207 sha256=e6497bf56a59ba88f8ecebd1814b055f98d66620fb9c6a9a2ed7235ccfde034d
  Stored in directory: /root/.cache/pip/wheels/5a/88/aa/d84b2cf1bad6b273cbf661640141a82c7b9f496e024f80aac0
Successfully built fairscale


Imports

In [5]:
from labml import experiment
from labml_nn.transformers.gpt import Configs

Create an experiment

In [6]:
experiment.create(name="gpt")

Initialize [GPT configurations](https://nn.labml.ai/transformers/gpt/)

In [7]:
conf = Configs()

Set experiment configurations and assign a configurations dictionary to override configurations

In [8]:
experiment.configs(conf, {
    # Use character level tokenizer
    'tokenizer': 'character',
    # Prompt separator is blank
    'prompt_separator': '',
    # Starting prompt for sampling
    'prompt': 'It is ',
    # Use Tiny Shakespeare dataset
    'text': 'tiny_shakespeare',

    # Use a context size of $128$
    'seq_len': 128,
    # Train for $32$ epochs
    'epochs': 32,
    # Batch size $128$
    'batch_size': 128,
    # Switch between training and validation for $10$ times
    # per epoch
    'inner_iterations': 10,

    # Transformer configurations
    'transformer.d_model': 512,
    'transformer.ffn.d_ff': 2048,
    'transformer.n_heads': 8,
    'transformer.n_layers': 6
})

Set PyTorch models for loading and saving

Start the experiment and run the training loop.

In [9]:
# Start the experiment
with experiment.start():
    conf.run()

AttributeError: module 'labml.tracker' has no attribute 'set_text'

### Testing the model
Now that training is complete, let's use the model to generate some text from a custom prompt.

In [14]:
import torch

def generate_text(prompt, length=100):
    conf.model.eval()
    # Access the tokenizer from the text configuration
    tokenizer = conf.text.tokenizer

    with torch.no_grad():
        # Encode the prompt
        ids = tokenizer.encode(prompt).to(conf.device)

        for _ in range(length):
            # Get predictions for the last sequence
            output, _ = conf.model(ids[-conf.seq_len:].unsqueeze(-1))
            # Take the last character prediction
            next_id = output[-1].argmax(dim=-1).unsqueeze(0)
            # Append to sequence
            ids = torch.cat([ids, next_id])

        # Convert tensor to list for decoding
        return tokenizer.decode(ids.tolist())

# Try a custom prompt
test_prompt = "To be or not to be"
generated = generate_text(test_prompt, length=200)
print(generated)

TypeError: character_tokenizer() missing 1 required positional argument: 'x'